# Complete workflow: from rod, line and stroke to a simulated cast

This notebook runs steps 2 to 5 of the workflow in sequence. Each step
produces an input for the cast simulation at the end. The step notebooks
go into more detail.

The `flycastsim` workflow has five steps. Step 1 stands on its own, and
steps 2 to 4 each produce parameters that step 5 uses.

1. Simple 1-D model (`01_simple_1d_model.ipynb`)
2. Rod characteristics (`02_rod_characteristics.ipynb`)
3. Line and leader characteristics (`03_line_and_leader.ipynb`)
4. Casting parameters (`04_casting_parameters.ipynb`)
5. Simulate a fly cast (`05_simulate_cast.ipynb`)

`complete_workflow.ipynb` runs steps 2 to 5 in sequence.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from flycastsim import plot_cast_snapshots, plot_chord_comparison
from flycastsim.fem import (
    bundled_rig_path,
    cast1_stroke,
    load_component,
    simulate_cast1,
)
from flycastsim.rod import fitted_rods_table, load_fitted_rods

## Step 2: Rod characteristics

Pick a rod. The three fitted rods carry a stiffness profile EI(s) and a
mass profile μ(s) fitted to bend tests, a frequency test and the CCS
point.

In [ ]:
fitted_rods = {rod.id: rod for rod in load_fitted_rods()}
table = pd.DataFrame(fitted_rods_table(fitted_rods.values())).set_index("rod")
table[["label", "length [m]", "mass [g]", "swingweight [g m²]",
       "EI clamp [N m²]", "EI mid [N m²]"]]

In [ ]:
ROD_ID = "XP590"
rod = fitted_rods[ROD_ID]
print(f"Selected rod: {rod.label}, {rod.length:.2f} m, "
      f"swingweight {rod.swingweight_gm2:.0f} g m²")

## Step 3: Line and leader characteristics

Pick a line weight. The bundled fly line profile is scaled to the AFTM
weight, and the bundled leader is attached to it.

In [ ]:
LINE_WEIGHT = 5

components = bundled_rig_path("cast1").parent
line = load_component(components / "line.json", aftm_weight=LINE_WEIGHT)
leader = load_component(components / "leader.json")
for comp in (line, leader):
    print(f"{comp.kind:7s} {comp.length:5.2f} m, "
          f"{np.trapezoid(comp.m, comp.s) * 1000:5.2f} g")

## Step 4: Casting parameters

The stroke is the rod-butt angle over time. Cast #1 has its stroke built
into the simulation, so here it is inspected, not chosen: a stroke cannot
yet be loaded from a file or passed to the simulation.

In [ ]:
t_stroke = np.linspace(-0.4, 1.0, 701)
theta = cast1_stroke()
fig = go.Figure(go.Scatter(
    x=t_stroke, y=np.degrees([theta(ti) for ti in t_stroke])))
fig.add_vline(x=0.0, line_dash="dash", annotation_text="RSP")
fig.update_layout(title="Cast #1 stroke",
                  xaxis_title="time relative to RSP [s]",
                  yaxis_title="rod-butt angle from horizontal [deg]")
fig

## Step 5: Simulate a fly cast

The rod from step 2 and the line weight from step 3 go into the Cast #1
simulation. The fitted rods have no measured diameter, which only matters
for air drag, so the diameter is borrowed from the Cast #1 rod taper.

In [ ]:
t, X, Y, s, chord, deflection, _, rod_tip = simulate_cast1(
    rod=rod.component, rod_diameter="cast1", aftm_weight=LINE_WEIGHT)
plot_cast_snapshots(t, X, Y, rod_tip_index=rod_tip)

### Compare with the tabulated Cast #1 rod

The same cast with the original, tabulated rod. The stroke was fitted by
eye to that much stiffer rod, so the fitted rods bend more than the
footage shows.

In [ ]:
t0, X0, Y0, s0, chord0, deflection0, _, rod_tip0 = simulate_cast1(
    aftm_weight=LINE_WEIGHT)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t0, y=deflection0, name="tabulated Cast #1 rod"))
fig.add_trace(go.Scatter(x=t, y=deflection, name=f"fitted {ROD_ID}"))
fig.add_vline(x=0.0, line_dash="dash", annotation_text="RSP")
fig.update_layout(title="Rod tip deflection",
                  xaxis_title="time relative to RSP [s]",
                  yaxis_title="tip deflection [m]")
fig

The chord length of the tabulated rod against the curve measured from the
footage:

In [ ]:
plot_chord_comparison(t0, chord0)

In [ ]:
# Compare the rod loading, which happens before RSP (t < 0).
summary = pd.DataFrame({
    "rod": ["tabulated Cast #1 rod", f"fitted {ROD_ID}"],
    "largest tip deflection before RSP [m]": [
        np.abs(deflection0[t0 < 0]).max(), np.abs(deflection[t < 0]).max()],
    "shortest chord before RSP [m]": [
        chord0[t0 < 0].min(), chord[t < 0].min()],
}).set_index("rod")
summary.round(2)